# Map the VIIRS tiles and extract any area

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
import geopandas as gpd
from shapely.geometry import box

# --------------------------------------------------
# Load land mask (Natural Earth)
# --------------------------------------------------
shpfilename = shpreader.natural_earth(
    resolution="110m",
    category="physical",
    name="land"
)

land = gpd.read_file(shpfilename).to_crs("EPSG:4326")
land_union = land.geometry.union_all()

# --------------------------------------------------
# AOI grid: 15° x 15°
# --------------------------------------------------
lon_edges = [-180, -165, -150, -135, -120, -105, -90, -75,-60]
lat_edges = [75, 60, 45, 30, 15]

aois = []
aoi_id = 1
skipped = 0

for lat_max, lat_min in zip(lat_edges[:-1], lat_edges[1:]):
    for lon_min, lon_max in zip(lon_edges[:-1], lon_edges[1:]):
        tile = box(lon_min, lat_min, lon_max, lat_max)

        # --------------------------------------------------
        # Skip water-only tiles
        # --------------------------------------------------
        if not tile.intersects(land_union) :
            continue
        if aoi_id == 11 and skipped == 0 :
            skipped=1
            continue  
        if aoi_id == 16 :
            aoi_id = 17
        if aoi_id == 22 and skipped == 1 :
            skipped=2
            continue       
  
        aois.append((aoi_id, tile))
        aoi_id += 1

lon_edges = [-105, -90, -75,-60,-45,-30]
lat_edges = [15,0,-15,-30,-45,-60]
skipped = 0

for lat_max, lat_min in zip(lat_edges[:-1], lat_edges[1:]):
    for lon_min, lon_max in zip(lon_edges[:-1], lon_edges[1:]):
        tile = box(lon_min, lat_min, lon_max, lat_max)

        # --------------------------------------------------
        # Skip water-only tiles
        # --------------------------------------------------
        if not tile.intersects(land_union) :
            continue
        if aoi_id == 34 and skipped == 0 :
            skipped=1
            continue       
  
        aois.append((aoi_id, tile))
        aoi_id += 1
        
lon_edges = [-15, -0, 15,30,45,60,75,90,105,120,135,150,165,180]
lat_edges = [75, 60, 45, 30]
skipped = 0

for lat_max, lat_min in zip(lat_edges[:-1], lat_edges[1:]):
    for lon_min, lon_max in zip(lon_edges[:-1], lon_edges[1:]):
        tile = box(lon_min, lat_min, lon_max, lat_max)

        # --------------------------------------------------
        # Skip water-only tiles
        # --------------------------------------------------
        if not tile.intersects(land_union) :
            continue
            
        if aoi_id == 50 :
            aoi_id = 52   
  
        aois.append((aoi_id, tile))
        aoi_id += 1

aoi_id = 82
lon_edges = [-30,-15, -0, 15,30,45,60,75,90,105,120,135,150,165,180]
lat_edges = [30,15,0,-15,-30,-45,-60]
skipped = 0

for lat_max, lat_min in zip(lat_edges[:-1], lat_edges[1:]):
    for lon_min, lon_max in zip(lon_edges[:-1], lon_edges[1:]):
        tile = box(lon_min, lat_min, lon_max, lat_max)

        # --------------------------------------------------
        # Skip water-only tiles
        # --------------------------------------------------
        if not tile.intersects(land_union) :
            continue
         
        if aoi_id == 108 :
            aoi_id = 109

        if aoi_id == 113 and skipped == 0 :
            skipped=1
            continue  
        if aoi_id == 113 and skipped == 1 :
            skipped=2
            continue  
        if aoi_id == 121 and skipped == 2 :
            skipped=3
            continue       
        if aoi_id == 128 and skipped == 3 :
            skipped=4
            continue   
            
        aois.append((aoi_id, tile))
        aoi_id += 1
                    #lon_min, lat_min, lon_max, lat_max
aoi_id=16 ; tile = box (-60, 45, -45, 60 ) ; aois.append((aoi_id, tile))
aoi_id=50 ; tile = box (90, 75, 105, 90 ) ; aois.append((aoi_id, tile))
aoi_id=51 ; tile = box (105, 75, 120, 90 ) ; aois.append((aoi_id, tile))
aoi_id=81 ; tile = box (150, 30, 165, 45 ) ; aois.append((aoi_id, tile))
aoi_id=108 ; tile = box (75, -15, 90, 0 ) ; aois.append((aoi_id, tile))
aoi_id=129 ; tile = box (-150, 45, -135, 60 ) ; aois.append((aoi_id, tile))
aoi_id=130 ; tile = box (-60, 60, -45, 75 ) ; aois.append((aoi_id, tile))
aoi_id=131 ; tile = box (-45, 60, -30, 75 ) ; aois.append((aoi_id, tile))
aoi_id=132 ; tile = box (-30, 60, -15, 75 ) ; aois.append((aoi_id, tile))
aoi_id=133 ; tile = box (-160, 15, -150, 25 ) ; aois.append((aoi_id, tile))
aoi_id=134 ; tile = box (150, -15, 165, 0 ) ; aois.append((aoi_id, tile))
aoi_id=135 ; tile = box (165, -30, 180, -15 ) ; aois.append((aoi_id, tile))
aoi_id=136 ; tile = box (-180, -20, -165, -5 ) ; aois.append((aoi_id, tile))

# Order the AOIs
aois = sorted(aois, key=lambda x: x[0])


In [ ]:
# --------------------------------------------------
# Plot
# --------------------------------------------------
fig = plt.figure(figsize=(12, 8))
ax = plt.axes(projection=ccrs.PlateCarree())

ax.set_extent([-180, 180, -90, 90], crs=ccrs.PlateCarree())

ax.add_feature(cfeature.OCEAN, facecolor="aliceblue")
ax.add_feature(cfeature.LAND, facecolor="lightgray")
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linestyle=":", linewidth=0.5)

# --------------------------------------------------
# Draw AOIs with numbering
# --------------------------------------------------
for aoi_id, geom in aois:
    x, y = geom.exterior.xy
    ax.plot(x, y, color="red", linewidth=1.5)

    c = geom.centroid
    ax.text(
        c.x, c.y, str(aoi_id),
        ha="center", va="center",
        fontsize=8, fontweight="bold",
        transform=ccrs.PlateCarree()
    )

ax.set_title("VIIRS Flood Coverage AOIs 1–136 (15° × 15°, water-only skipped)", fontsize=14)
plt.show()

In [ ]:
import geopandas as gpd

# aois = [(AOI_ID, shapely.geometry), ...]

gdf_aois = gpd.GeoDataFrame(
    [{"AOI_ID": aoi_id, "geometry": geom} for aoi_id, geom in aois],
    crs="EPSG:4326"
)

# Sort and validate
gdf_aois = gdf_aois.sort_values("AOI_ID").reset_index(drop=True)

print("Number of AOIs:", len(gdf_aois))  # should be 136

# Save to file (recommended format)
gdf_aois.to_file("/perm/pad/flood_cases/VIIRS/VIIRS_AOIs_official.geojson", driver="GeoJSON")

In [ ]:
#Compare to the AOIs of VIIRS website
#aoi="/perm/pad/flood_cases/VIIRS/aoi.png"
#from PIL import Image
#import matplotlib.pyplot as plt
#plt.figure(figsize=(18, 10))
#img = Image.open(aoi)
#plt.imshow(img)
#plt.axis("off")

In [ ]:
## Extract VIIRS composite of AOIs for a given area

In [ ]:
#Read the AOIs from the saved file
import os
import requests
import rasterio
import zipfile
from rasterio.merge import merge
import geopandas as gpd
from shapely.geometry import box
import matplotlib.pyplot as plt

gdf_aois = gpd.read_file("/perm/pad/flood_cases/VIIRS/VIIRS_AOIs_official.geojson")

# Sanity check
assert gdf_aois.crs.to_string() == "EPSG:4326"
print("AOIs loaded:", len(gdf_aois))

In [ ]:
#### For a given area find the AOIs intersecting
from shapely.geometry import box

# area = [lat_min, lon_min, lat_max, lon_max]
area = [44, -126, 50, -120]  # Oregon

flood_case="Pakistan"
date = 20220830
area = [22, 66, 31, 72] # Pakistan

flood_case="EmiliaRomagna"
date = 20230521
area = [45, 11, 44, 13] #EmiliaRomagna

flood_case="SE_Asia" 
date = 20200716
area = [5, 60, 40, 95]  # SE_Asia 2/7/2020
area = [20, 80, 30, 95]  # SE_Asia 2/7/2020

flood_case="Amazon"
date = 20210616
area = [-17, -80, 5, -50] # Amazon

flood_case="US"
date = 20250704
area = [25, -107, 36.5, -93] # US Houston

flood_case="Australia"
date = 20250114
date = 20251231
date = 20260111
area = [-25, 135, -10, 155]  # Australia

flood_case="SouthAfrica"
date = 20260124
area = [-15, 10, -35, 40]
area = [-20, 30, -28, 38] #South Africa

flood_case="Spain"
date = 20260205
area = [31.5, -10, 40, 0] #Spain

flood_case="France"
date = 20260215
area = [42, -2, 46, 2] #France

flood_case="Persia"
area = [50, 35, 20, 65]
date = 20260329

flood_case="Yangtze"
area = [28, 105, 38, 125]  # Yangtze 
date = 20200722

area_geom = box(
    area[1], area[0],  # lon_min, lat_min
    area[3], area[2]   # lon_max, lat_max
)

hits = gdf_aois[gdf_aois.intersects(area_geom)]

aoi_ids = sorted(hits["AOI_ID"].tolist())
print("Intersecting AOIs:", aoi_ids)

In [ ]:
import requests
from bs4 import BeautifulSoup
import os

def find_remote_file(base_url, pattern):
    r = requests.get(base_url, timeout=60)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    for link in soup.find_all("a"):
        href = link.get("href", "")
        if pattern in href:
            return href

    return None

In [ ]:
OUTDIR  = "/perm/pad/flood_cases/VIIRS"

In [ ]:
NRT=0
if (NRT == 1 ):
    def download_tile(aoi_id, date):
        NRT_VIIRS_site="https://floodlight.ssec.wisc.edu/composite"
        filename = f"RIVER-FLDglobal-composite1_{date}_000000.part{aoi_id:03d}.tif"
        base_url = f"{NRT_VIIRS_site}/"
        outpath = os.path.join(OUTDIR, filename)

        if filename is None:
            print("⚠ No matching remote file")
        else:
            url = base_url + filename
            print("Downloading:", url)
    
        if os.path.exists(outpath):
            print(f"✔ Already downloaded: {filename}")
            return outpath
    
        print(f"⬇ Downloading: {filename}")
        r = requests.get(url, timeout=60)
        r.raise_for_status()
    
        with open(outpath, "wb") as f:
            f.write(r.content)
    
        return outpath

if (NRT == 0 ):
    def download_tile(aoi_id, date):
        VIIRS_site="https://jpssflood.gmu.edu/downloads/pub"
        base_url = f"{VIIRS_site}/{date}/tif/"
        pattern = f"_005day_{aoi_id:03d}"
        filename = find_remote_file(base_url, pattern)
        outpath = os.path.join(OUTDIR, filename)
    
        zip_name = filename
        tif_name = zip_name.replace(".zip", "")
    
        url = f"{VIIRS_site}/{date}/tif/{zip_name}"
        zip_path = os.path.join(OUTDIR, zip_name)
        tif_path = os.path.join(OUTDIR, tif_name)
    
        # --------------------------------------------------
        # If already extracted, return TIF
        # --------------------------------------------------
        if os.path.exists(tif_path):
            print(f"✔ Already extracted: {tif_name}")
            return tif_path
    
        # --------------------------------------------------
        # If ZIP exists but not extracted, unzip
        # --------------------------------------------------
        if os.path.exists(zip_path):
            print(f"📦 Unzipping existing: {zip_name}")
            try:
                with zipfile.ZipFile(zip_path, "r") as z:
                    z.extractall(OUTDIR)
                return tif_path
            except zipfile.BadZipFile:
                print(f"⚠ Corrupted ZIP: {zip_name}")
                return None
    
        # --------------------------------------------------
        # Download ZIP
        # --------------------------------------------------
        print(f"⬇ Downloading: {zip_name}")
        try:
            r = requests.get(url, timeout=60)
            if r.status_code != 200:
                print(f"⚠ Missing on server: {zip_name}")
                return None
    
            with open(zip_path, "wb") as f:
                f.write(r.content)
    
            # --------------------------------------------------
            # Unzip
            # --------------------------------------------------
            with zipfile.ZipFile(zip_path, "r") as z:
                z.extractall(OUTDIR)
    
            return tif_path
    
        except requests.RequestException as e:
            print(f"⚠ Download failed for {zip_name}: {e}")
            return None
        except zipfile.BadZipFile:
            print(f"⚠ Corrupted ZIP after download: {zip_name}")
            return None

In [ ]:
#Create a mosaic of the tiles downloaded for a single date
tile_files = [
    fp for fp in (download_tile(aoi_id,date) for aoi_id in aoi_ids)
    if fp is not None
]

In [ ]:
print(f"\nNumber of input tiles: {len(tile_files)}")
srcs = [rasterio.open(fp) for fp in tile_files]

mosaic, transform = merge(srcs)

meta = srcs[0].meta.copy()
meta.update({
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": transform
})

In [ ]:
import cartopy.crs as ccrs
import numpy as np

# --------------------------------------------------
# Compute geographic extent
# --------------------------------------------------
height, width = mosaic.shape[1], mosaic.shape[2]

lon_min = transform.c
lon_max = transform.c + transform.a * width
lat_max = transform.f
lat_min = transform.f + transform.e * height  # e is negative

extent = [lon_min, lon_max, lat_min, lat_max]

fig = plt.figure(figsize=(10, 8))
ax = plt.axes(projection=ccrs.PlateCarree())

ax.set_extent(extent, crs=ccrs.PlateCarree())

im = ax.imshow(
    mosaic[0],
    cmap="Blues",
    extent=extent,
    transform=ccrs.PlateCarree(),
    origin="upper"
)

ax.coastlines(resolution="10m", linewidth=0.8)
ax.gridlines(draw_labels=True)

plt.colorbar(im, ax=ax, label="VIIRS Flood Composite")
ax.set_title("VIIRS Flood Composite ("+flood_case+")")

plt.show()


In [ ]:
from rasterio.mask import mask
from shapely.geometry import box
import geopandas as gpd

# Build AOI geometry
area_geom = box(
    area[1], area[0],  # lon_min, lat_min
    area[3], area[2]   # lon_max, lat_max
)

# Create temporary dataset-like object
meta.update({"transform": transform})

with rasterio.io.MemoryFile() as memfile:
    with memfile.open(**meta) as dataset:
        dataset.write(mosaic)
        clipped, clipped_transform = mask(
            dataset,
            [area_geom],
            crop=True
        )
        
h, w = clipped.shape[1], clipped.shape[2]

lon_min = clipped_transform.c
lon_max = clipped_transform.c + clipped_transform.a * w
lat_max = clipped_transform.f
lat_min = clipped_transform.f + clipped_transform.e * h

extent = [lon_min, lon_max, lat_min, lat_max]

In [ ]:
fig = plt.figure(figsize=(12, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

ax.set_extent(extent, crs=ccrs.PlateCarree())

im = ax.imshow(
    clipped[0],
    cmap="Blues",
    extent=extent,
    transform=ccrs.PlateCarree(),
    origin="upper"
)

ax.coastlines(resolution="10m", linewidth=0.5)
ax.gridlines(draw_labels=True)

plt.colorbar(im, ax=ax, label="VIIRS Flood Composite")
ax.set_title("VIIRS Flood Composite ("+flood_case+" - "+str(date)+")")
plt.show()

In [ ]:
import numpy as np

vals = np.unique(clipped[0])
print(vals)

In [ ]:
import numpy as np

data = clipped[0]

water_perm = (data == 17)
water_seasonal = (data == 20)
cloud = (data == 30)
water = (data == 99)

# Flood masks by confidence
flood_low = (data >= 130) & (data < 160)
flood_med = (data >= 160) & (data < 190)
flood_high = (data >= 190)
flood = data >= 160   # or >= 170 depending on conservativeness
flood_masked = np.where(cloud, np.nan, flood)
flood_masked = np.where(water_perm, np.nan, flood_masked)

In [ ]:
fig = plt.figure(figsize=(12, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

ax.set_extent(extent, crs=ccrs.PlateCarree())

im = ax.imshow(
    flood_masked,
    cmap="Blues",
    extent=extent,
    transform=ccrs.PlateCarree(),
    origin="upper"
)

ax.coastlines(resolution="10m", linewidth=0.5)
ax.gridlines(draw_labels=True)

plt.colorbar(im, ax=ax, label="VIIRS Flood Composite")
ax.set_title("VIIRS Flood Composite ("+flood_case+" - "+str(date)+")")

plt.show()

In [ ]:
path_obs  = "/perm/pad/flood_cases/"+flood_case+"_flood_obs_VIIRS.tif"
path_mask = "/perm/pad/flood_cases/"+flood_case+"_flood_mask_VIIRS.tif"

In [ ]:
flood_uint8 = np.zeros_like(data, dtype=np.uint8)
flood_uint8[flood] = 1   # 1 = flooded, 0 = not flooded
mask_uint8 = np.ones_like(data, dtype=np.uint8)
mask_uint8[cloud] = 0   # 1 = wat
mask_uint8[water] = 0   # 1 = wat


In [ ]:
import rasterio
meta = {
    "driver": "GTiff",
    "height": flood_uint8.shape[0],
    "width": flood_uint8.shape[1],
    "count": 1,
    "dtype": "uint8",
    "crs": "EPSG:4326",
    "transform": transform,
    "nodata": 0,
    "compress": "LZW"
}

with rasterio.open(path_obs, "w", **meta) as dst:
    dst.write(flood_uint8, 1)

print("Saved:", path_obs)

meta = {
    "driver": "GTiff",
    "height": mask_uint8.shape[0],
    "width": mask_uint8.shape[1],
    "count": 1,
    "dtype": "uint8",
    "crs": "EPSG:4326",
    "transform": transform,
    "nodata": 0,
    "compress": "LZW"
}

with rasterio.open(path_mask, "w", **meta) as dst:
    dst.write(mask_uint8, 1)

print("Saved:", path_mask)

In [ ]:
import rasterio
import matplotlib.pyplot as plt
from rasterio.enums import Resampling

def read_downsampled(path, factor=1):
    with rasterio.open(path) as ds:
        arr = ds.read(
            1,
            out_shape=(
                1,
                ds.height // factor,
                ds.width // factor
            ),
            resampling=Resampling.nearest
        )
    return arr

obs  = read_downsampled(path_obs, factor=1)
mask = read_downsampled(path_mask, factor=1)

# Plot
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.imshow(obs, cmap="Blues",vmin=0,vmax=1)
plt.title(flood_case+" ("+str(date)+")"+" — Flood Observations")
plt.colorbar(fraction=0.046, pad=0.04)

plt.subplot(1, 2, 2)
plt.imshow(mask, cmap="Reds", vmin=0,vmax=1)
plt.title(flood_case+" ("+str(date)+")"+" — Flood Mask")
plt.colorbar(fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()
